In [1]:
pip install beautifulsoup4 lxml


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from bs4 import BeautifulSoup

In [4]:
import requests
from bs4 import BeautifulSoup

url = "https://books.toscrape.com"
r = requests.get(url, timeout = 10)
r.raise_for_status()
soup = BeautifulSoup(r.text, "lxml")

print(soup.title.text)


    All products | Books to Scrape - Sandbox



In [5]:
books = soup.select("article.product_pod")
print(f"이 페이지에 {len(books)}권")

for book in books[:3]:
    title = book.h3.a["title"]
    price = book.select_one(".price_color").text
    avail = book.select_one(".availability").text.strip()
    print(f"{title:40} {price:8} {avail}")

이 페이지에 20권
A Light in the Attic                     Â£51.77  In stock
Tipping the Velvet                       Â£53.74  In stock
Soumission                               Â£50.10  In stock


In [7]:
import time, requests
from bs4 import BeautifulSoup

base = "https://books.toscrape.com/catalogue/page-{n}.html"
all_books: list[dict] = []
for n in range(1, 6):
    r = requests.get(base.format(n=n), timeout = 10)
    soup = BeautifulSoup(r.text, "lxml")
    for book in soup.select("article.product_pod"):
        all_books.append({
            "title": book.h3.a["title"],
            "price": book.select_one(".price_color").text, 
        })
    time.sleep(1)
print(len(all_books), "권 수집")

100 권 수집


In [9]:
r = requests.get("https://quotes.toscrape.com", timeout = 10)
soup = BeautifulSoup(r.text, "lxml")

for q in soup.select("div.quote")[:2]:
    text = q.select_one("span.text").text
    author = q.select_one("small.author").text
    tags = [t.text for t in q.select("a.tag")]
    print(f"{author}: {text}")
    print(f"    태그: {tags}")

Albert Einstein: “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
    태그: ['change', 'deep-thoughts', 'thinking', 'world']
J.K. Rowling: “It is our choices, Harry, that show what we truly are, far more than our abilities.”
    태그: ['abilities', 'choices']


In [10]:
import csv, time, requests
from bs4 import BeautifulSoup

rows: list[dict] = []
for page in range(1, 4):
    r = requests.get(f"https://quotes.toscrape.com/page/{page}", timeout = 10)
    soup = BeautifulSoup(r.text, "lxml")
    for q in soup.select("div.quote"):
        rows.append({
            "author": q.select_one("small.author").text,
            "quote": q.select_one("span.text").text,
            "tags": ",".join(t.text for t in q.select("a.tag")),
        })
    time.sleep(1)
with open("quotes.csv", "w", encoding = "utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames = ["author", "quote", "tags"])
    writer.writeheader(); writer.writerows(rows)
    

In [11]:
params = {"searchKeyword": "언어", "pageSize": 10}
headers = {"User-Agent": "Mozilla/5.0"}
r = requests.get(
    "https://stdict.korean.go.kr/search/searchResult.do",
    params = params, headers = headers, timeout = 10,
)
print(r.status_code, len(r.text))

200 42779


In [12]:
soup = BeautifulSoup(r.text, "lxml")

for item in soup.select("ul.search_list li"):
    word = item.select_one(".word_head").get_text(strip = True)
    pos = item.select_one(".word_class").get_text(strip = True)
    sense = item.select_one(".search_sub").get_text(" ", strip = True)
    print(f"{word} ({pos}) --- {sense[:50]}")

In [13]:
from urllib.robotparser import RobotFileParser

rp = RobotFileParser()
rp.set_url("https://books.toscrape.com/robot.txt")
rp.read()

rp.can_fetch("*",
             "https://books.toscrape.com/catalogue/page-1.html")

True

In [14]:
from pathlib import Path
import hashlib, requests

CACHE = Path("cache"); CACHE.mkdir(exist_ok = True)

def get_cached(url: str) -> str:
    key = hashlib.md5(url.encode()).hexdigest()
    path = CACHE / f"{key}.html"
    if path.exist():
        return path.read_text(encoding="utf-8")
    r = requests.get(url, timeout = 10)
    r.raise_for_status()
    path.write_text(r.text, encoding="utf-8")
    return r.text